## 01b — Retention Cohorts

Part 1(b) of the assignment.

Players are grouped by install week and retention is measured at:
- Day 1
- Day 7
- Day 14
- Day 30

A cohort/milestone is reported only when the complete observation window for that milestone is available. Recent cohorts without sufficient follow-up remain unobserved rather than being treated as zero retention.

In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.db import get_connection

conn = get_connection()

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

print("Database connected successfully.")

Database connected successfully.


In [2]:
conn.execute("""
CREATE INDEX IF NOT EXISTS idx_player_day_device_date
ON player_day(device_id, activity_date)
""")

conn.execute("""
CREATE INDEX IF NOT EXISTS idx_profile_install
ON player_profile(install_date)
""")

conn.commit()

print("Indexes ready.")

Indexes ready.


In [3]:
sql_path = "../sql/1b_retention_cohorts.sql"

with open(sql_path, "r", encoding="utf-8") as f:
    query = f.read()

df_01b = pd.read_sql_query(query, conn)

print("Shape:", df_01b.shape)
df_01b

Shape: (18, 6)


,cohort_week,cohort_size,retention_d1,retention_d7,retention_d14,retention_d30
0,2026-01-01,2129,0.4612,0.2518,0.1902,0.1249
1,2026-01-08,2272,0.4789,0.2672,0.2011,0.1536
2,2026-01-15,2274,0.4767,0.2573,0.1825,0.1359
3,2026-01-22,2166,0.4746,0.2604,0.2004,0.1334
4,2026-01-29,2271,0.4866,0.2457,0.2149,0.1480
5,2026-02-05,2262,0.4757,0.2759,0.2042,0.1508
6,2026-02-12,2333,0.5011,0.2435,0.2002,0.1449
7,2026-02-19,2360,0.4606,0.2504,0.1975,0.1403
8,2026-02-26,2976,0.4782,0.2634,0.1872,0.1341
9,2026-03-05,3466,0.4735,0.2519,0.1959,0.1420


In [4]:
print("Columns:")
print(df_01b.columns.tolist())

print("\nMissing values by column:")
print(df_01b.isna().sum())

print("\nTotal cohort rows:", len(df_01b))

Columns:
['cohort_week', 'cohort_size', 'retention_d1', 'retention_d7', 'retention_d14', 'retention_d30']

Missing values by column:
cohort_week      0
cohort_size      0
retention_d1     1
retention_d7     2
retention_d14    3
retention_d30    6
dtype: int64

Total cohort rows: 18


In [5]:
retention_cols = [
    col for col in df_01b.columns
    if "retention" in col.lower()
]

print("Retention columns:", retention_cols)

for col in retention_cols:
    valid = df_01b[col].dropna()

    print(
        f"{col}: "
        f"min={valid.min():.4f}, "
        f"max={valid.max():.4f}, "
        f"observed={len(valid)}, "
        f"missing={df_01b[col].isna().sum()}"
    )

    assert valid.between(0, 1).all(), \
        f"{col} contains values outside [0, 1]"

print("\nRetention range checks passed.")

Retention columns: ['retention_d1', 'retention_d7', 'retention_d14', 'retention_d30']
retention_d1: min=0.4606, max=0.5011, observed=17, missing=1
retention_d7: min=0.2435, max=0.2759, observed=16, missing=2
retention_d14: min=0.1825, max=0.2149, observed=15, missing=3
retention_d30: min=0.1249, max=0.1536, observed=12, missing=6

Retention range checks passed.


In [6]:
print("Retention availability by milestone:")

for col in retention_cols:
    print(
        f"{col}: "
        f"{df_01b[col].notna().sum()} observed, "
        f"{df_01b[col].isna().sum()} not yet observable"
    )

Retention availability by milestone:
retention_d1: 17 observed, 1 not yet observable
retention_d7: 16 observed, 2 not yet observable
retention_d14: 15 observed, 3 not yet observable
retention_d30: 12 observed, 6 not yet observable


In [7]:
output_path = "../outputs/tables/01b_retention_cohorts.csv"

df_01b.to_csv(output_path, index=False)

print(f"Saved successfully: {output_path}")

Saved successfully: ../outputs/tables/01b_retention_cohorts.csv


In [8]:
saved = pd.read_csv("../outputs/tables/01b_retention_cohorts.csv")

print("Notebook rows:", len(df_01b))
print("Saved rows:", len(saved))
print("Same columns:", saved.columns.tolist() == df_01b.columns.tolist())

assert len(saved) == len(df_01b)
assert saved.columns.tolist() == df_01b.columns.tolist()

print("\nOutput file verified successfully.")

Notebook rows: 18
Saved rows: 18
Same columns: True

Output file verified successfully.


In [9]:
conn.close()
print("Database connection closed.")

Database connection closed.
